In [ ]:
import os
import re
import SimpleITK as sitk
import numpy as np
from scipy.ndimage import binary_fill_holes


def largest_connected_component(binary_img):
    cc = sitk.ConnectedComponent(binary_img)
    relabel = sitk.RelabelComponent(cc, sortByObjectSize=True)
    largest = sitk.Equal(relabel, 1)
    return sitk.Cast(largest, sitk.sitkUInt8)


def fill_holes_slice_by_slice(binary_img):
    arr = sitk.GetArrayFromImage(binary_img)  # z, y, x
    filled = np.zeros_like(arr, dtype=np.uint8)

    for z in range(arr.shape[0]):
        filled[z] = binary_fill_holes(arr[z]).astype(np.uint8)

    out = sitk.GetImageFromArray(filled)
    out.CopyInformation(binary_img)
    return sitk.Cast(out, sitk.sitkUInt8)


def touches_border(mask2d, margin=1):
    h, w = mask2d.shape

    if mask2d[:margin, :].any():
        return True
    if mask2d[h - margin:, :].any():
        return True
    if mask2d[:, :margin].any():
        return True
    if mask2d[:, w - margin:].any():
        return True

    return False


def remove_truncated_slices_from_mask(mask_img, reference_body_img, border_margin=1, min_area_px=2000):
    mask = sitk.GetArrayFromImage(mask_img)
    body = sitk.GetArrayFromImage(reference_body_img)

    cleaned = mask.copy()
    valid_slices = np.zeros(body.shape[0], dtype=bool)

    for z in range(body.shape[0]):
        sl = body[z] > 0
        area = sl.sum()

        if area < min_area_px:
            cleaned[z] = 0
            valid_slices[z] = False
            continue

        if touches_border(sl, margin=border_margin):
            cleaned[z] = 0
            valid_slices[z] = False
            continue

        valid_slices[z] = True

    out = sitk.GetImageFromArray(cleaned.astype(np.uint8))
    out.CopyInformation(mask_img)
    return sitk.Cast(out, sitk.sitkUInt8), valid_slices


def crop_to_original_size(padded_img, original_img, pad_voxels):
    arr = sitk.GetArrayFromImage(padded_img)   # z, y, x
    orig_arr = sitk.GetArrayFromImage(original_img)

    z0, y0, x0 = pad_voxels, pad_voxels, pad_voxels
    z1 = z0 + orig_arr.shape[0]
    y1 = y0 + orig_arr.shape[1]
    x1 = x0 + orig_arr.shape[2]

    cropped = arr[z0:z1, y0:y1, x0:x1]

    out = sitk.GetImageFromArray(cropped.astype(np.uint8))
    out.CopyInformation(original_img)
    return sitk.Cast(out, sitk.sitkUInt8)


def remove_table_slice_by_slice(binary_img, bottom_margin=3, min_component_area=500):
    """
    Elimina componentes 2D que tocan el borde inferior de cada corte axial.
    Esto ayuda a quitar la camilla antes del closing 3D.

    Parámetros:
    - bottom_margin: nº de píxeles desde el borde inferior considerados "zona de contacto"
    - min_component_area: descarta componentes muy pequeñas
    """
    arr = sitk.GetArrayFromImage(binary_img)  # z, y, x
    cleaned = np.zeros_like(arr, dtype=np.uint8)

    for z in range(arr.shape[0]):
        slice_arr = arr[z].astype(np.uint8)

        if slice_arr.sum() == 0:
            continue

        slice_img = sitk.GetImageFromArray(slice_arr)
        cc = sitk.ConnectedComponent(slice_img)
        cc_arr = sitk.GetArrayFromImage(cc)

        labels = np.unique(cc_arr)
        labels = labels[labels > 0]

        valid_components = []

        for lab in labels:
            comp = (cc_arr == lab)
            area = comp.sum()

            if area < min_component_area:
                continue

            # Si toca el borde inferior, muy probablemente es camilla o estructura no deseada
            if comp[-bottom_margin:, :].any():
                continue

            valid_components.append((lab, area))

        if len(valid_components) == 0:
            continue

        # Nos quedamos con la componente válida más grande de ese slice
        best_label = max(valid_components, key=lambda x: x[1])[0]
        cleaned[z][cc_arr == best_label] = 1

    out = sitk.GetImageFromArray(cleaned.astype(np.uint8))
    out.CopyInformation(binary_img)
    return sitk.Cast(out, sitk.sitkUInt8)


def create_shell_from_tac_robust(
    tac_path,
    body_threshold=-350,
    closing_radius=1,
    erosion_radius=1,
    pad_voxels=10,
    border_margin=1,
    min_area_px=2000,
    table_bottom_margin=3,
    table_min_component_area=500
):
    img = sitk.ReadImage(tac_path)

    img_padded = sitk.ConstantPad(
        img,
        [pad_voxels] * 3,
        [pad_voxels] * 3,
        -1024
    )

    # 1) Umbral inicial del cuerpo
    body = sitk.BinaryThreshold(
        img_padded,
        lowerThreshold=body_threshold,
        upperThreshold=3000,
        insideValue=1,
        outsideValue=0
    )

    # 2) Eliminar camilla slice a slice ANTES de cerrar en 3D
    body = remove_table_slice_by_slice(
        body,
        bottom_margin=table_bottom_margin,
        min_component_area=table_min_component_area
    )

    # 3) Limpieza morfológica
    body = sitk.BinaryMorphologicalClosing(body, [closing_radius] * 3)
    body = largest_connected_component(body)
    body = fill_holes_slice_by_slice(body)
    body = largest_connected_component(body)

    # 4) Quitar cortes truncados
    body_trimmed, valid_slices = remove_truncated_slices_from_mask(
        body,
        body,
        border_margin=border_margin,
        min_area_px=min_area_px
    )

    # 5) Crear shell = cuerpo - cuerpo erosionado
    body_eroded = sitk.BinaryErode(body_trimmed, [erosion_radius] * 3)

    skin = sitk.Subtract(body_trimmed, body_eroded)
    skin = sitk.BinaryThreshold(skin, 1, 1, 1, 0)

    skin, _ = remove_truncated_slices_from_mask(
        skin,
        body_trimmed,
        border_margin=border_margin,
        min_area_px=min_area_px
    )

    # 6) Recortar al tamaño original del TAC
    skin_cropped = crop_to_original_size(skin, img, pad_voxels)

    return skin_cropped


def replace_last_label_without_overwriting(
    tac_path,
    labelmap_path,
    output_labelmap_path,
    body_threshold=-350,
    closing_radius=1,
    erosion_radius=1,
    pad_voxels=10,
    border_margin=1,
    min_area_px=2000,
    table_bottom_margin=3,
    table_min_component_area=500
):
    # Crear nueva shell desde el TAC
    shell = create_shell_from_tac_robust(
        tac_path=tac_path,
        body_threshold=body_threshold,
        closing_radius=closing_radius,
        erosion_radius=erosion_radius,
        pad_voxels=pad_voxels,
        border_margin=border_margin,
        min_area_px=min_area_px,
        table_bottom_margin=table_bottom_margin,
        table_min_component_area=table_min_component_area
    )
    shell_arr = sitk.GetArrayFromImage(shell)

    # Leer labelmap original
    labelmap = sitk.ReadImage(labelmap_path)
    label_arr = sitk.GetArrayFromImage(labelmap)

    if label_arr.shape != shell_arr.shape:
        raise ValueError(
            f"Dimensiones distintas.\n"
            f"Labelmap: {label_arr.shape}\n"
            f"Shell:    {shell_arr.shape}"
        )

    labels = np.unique(label_arr)
    labels = labels[labels > 0]

    if len(labels) == 0:
        raise ValueError("El labelmap no contiene etiquetas > 0.")

    last_label = labels.max()
    print(f"  Última etiqueta detectada: {last_label}")

    # Borrar la antigua última etiqueta
    label_arr[label_arr == last_label] = 0

    # Insertar nueva shell SIN sobreescribir otras estructuras
    label_arr[(shell_arr > 0) & (label_arr == 0)] = last_label

    # Guardar en nueva carpeta
    new_labelmap = sitk.GetImageFromArray(label_arr.astype(np.uint16))
    new_labelmap.CopyInformation(labelmap)

    sitk.WriteImage(new_labelmap, output_labelmap_path)
    print(f"  Guardado en: {output_labelmap_path}")


def extract_patient_number(filename):
    """
    Extrae el número de paciente desde nombres tipo:
    - Paciente48_pre.nrrd
    - PacienteEtiquetas48_labelmap.nrrd
    """
    match = re.search(r'(\d+)', filename)
    return match.group(1) if match else None


def process_all_patients(
    tacs_folder,
    labelmaps_folder,
    output_folder,
    body_threshold=-350,
    closing_radius=1,
    erosion_radius=1,
    pad_voxels=10,
    border_margin=1,
    min_area_px=2000,
    table_bottom_margin=3,
    table_min_component_area=500
):
    os.makedirs(output_folder, exist_ok=True)

    tac_files = [f for f in os.listdir(tacs_folder) if f.lower().endswith(".nrrd")]
    labelmap_files = [f for f in os.listdir(labelmaps_folder) if f.lower().endswith(".nrrd")]

    tac_dict = {}
    for f in tac_files:
        num = extract_patient_number(f)
        if num is not None:
            tac_dict[num] = f

    label_dict = {}
    for f in labelmap_files:
        num = extract_patient_number(f)
        if num is not None:
            label_dict[num] = f

    common_patients = sorted(set(tac_dict.keys()) & set(label_dict.keys()), key=int)

    print(f"Pacientes emparejados encontrados: {len(common_patients)}")

    if len(common_patients) == 0:
        print("No se encontraron pacientes emparejados entre TACs y labelmaps.")
        return

    for num in common_patients:
        tac_file = tac_dict[num]
        label_file = label_dict[num]

        tac_path = os.path.join(tacs_folder, tac_file)
        labelmap_path = os.path.join(labelmaps_folder, label_file)
        output_labelmap_path = os.path.join(output_folder, label_file)

        print(f"\nProcesando paciente {num}")
        print(f"  TAC:      {tac_file}")
        print(f"  Labelmap: {label_file}")

        try:
            replace_last_label_without_overwriting(
                tac_path=tac_path,
                labelmap_path=labelmap_path,
                output_labelmap_path=output_labelmap_path,
                body_threshold=body_threshold,
                closing_radius=closing_radius,
                erosion_radius=erosion_radius,
                pad_voxels=pad_voxels,
                border_margin=border_margin,
                min_area_px=min_area_px,
                table_bottom_margin=table_bottom_margin,
                table_min_component_area=table_min_component_area
            )
        except Exception as e:
            print(f"  ERROR en paciente {num}: {e}")


process_all_patients(
    tacs_folder=r"C:\Users\iredondo\Documents\TFG\TACS",
    labelmaps_folder=r"C:\Users\iredondo\Documents\TFG\salida",
    output_folder=r"C:\Users\iredondo\Documents\TFG\PIEL",
    body_threshold=-350,
    closing_radius=1,
    erosion_radius=1,
    pad_voxels=10,
    border_margin=1,
    min_area_px=2000,
    table_bottom_margin=3,
    table_min_component_area=500
)